# Clase 5 · 04 · Registro, uso y monitoreo

Un run es un experimento; un **modelo registrado** es un producto. El **Model Registry** (en Databricks, dentro de Unity Catalog) le da al modelo un nombre estable, **versiones** numeradas y **alias** (`champion`, `challenger`) que indican cuál se usa. Así, el proceso que consume el modelo pide "el champion" y no un `run_id`: cambiar de modelo es mover el alias, sin tocar el código que lo usa.

En este notebook: registramos tu mejor run, lo usamos para **puntuar las transacciones nuevas** (inferencia batch) y, cuando llegan los contracargos reales, lo **monitoreamos**.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/ml_lab

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
spark.conf.set("spark.sql.session.timeZone", "UTC")
T = lambda name: qualified_table(config, name)
client = MlflowClient()

In [ ]:
assert spark.catalog.tableExists(T("ml_features")), "Falta ml_features. Ejecutá primero 00_preflight."
experiment = setup_experiment(spark, config)
data_version = table_version(spark, T("ml_features"))
print("Experimento:", experiment.name)

runs = my_runs(config, experiment.experiment_id)
assert len(runs), "No hay runs de modelos. Ejecutá los notebooks 01 y 02."
best = runs.iloc[0]
print(f"Mejor run por PR-AUC: {best.run}")
display(runs.head(1).drop(columns=["model_uri"]))

## 1. Registrar el modelo en Unity Catalog

El nombre tiene tres partes, como una tabla: `catálogo.esquema.modelo`. Cada vez que se registra el mismo nombre se crea una **versión** nueva. Después le asignamos el alias `champion`.

In [ ]:
model_name = registered_model_name(config, config.catalog)
registration = register_champion(best.model_uri, model_name)
if registration["registered"]:
    print(f"✅ Registrado: {model_name} versión {registration['version']} con alias 'champion'")
    versions = MlflowClient().search_model_versions(f"name = '{model_name}'")
    display(pd.DataFrame([{"versión": v.version, "run_id": v.run_id, "estado": v.status} for v in versions]))
else:
    print("⚠️  No se pudo registrar el modelo en Unity Catalog en este workspace.")
    print("   Motivo:", registration["error"])
    print("   La práctica sigue igual usando la URI del run:", registration["uri"])
model_uri = registration["uri"]

Si se registró, buscalo en **Catalog → tu catálogo → `bigdata_<tu_id>` → Models → `fraud_model`**: vas a ver las versiones, el alias y el vínculo con el run que lo generó (el **linaje**).

> Si el registro falló, la práctica continúa cargando el modelo desde el run (`runs:/...`). En la entrega contá qué error apareció: igual podés responder las preguntas.

## 2. Cargar el champion

Cargamos el modelo **por alias**. El umbral de decisión no está dentro del modelo: lo leemos de los parámetros del run, porque es una decisión de negocio que puede cambiar sin reentrenar.

In [ ]:
champion = mlflow.sklearn.load_model(model_uri)
features = FEATURE_SETS[best.features]
threshold = float(best.umbral)
version = registration["version"] or "sin_registro"
print(f"Modelo: {model_uri} · features: {best.features} · umbral: {threshold}")

## 3. Inferencia batch: puntuar las transacciones nuevas

Las transacciones del 7 de marzo (`split = 'nuevas'`) son las que el modelo nunca vio. Las puntuamos **sin mirar** su etiqueta y guardamos el resultado en `ml_predictions`, con todo lo necesario para auditar cada decisión: qué modelo, qué versión, qué run y qué umbral la produjo.

In [ ]:
nuevas = load_split(spark, config, "nuevas")
scores = champion.predict_proba(nuevas[features])[:, 1]
predictions = pd.DataFrame({
    "transaction_id": nuevas["transaction_id"],
    "score": scores.round(6),
    "alerta": (scores >= threshold).astype(int),
    "threshold": threshold,
    "model_name": model_name if registration["registered"] else "run",
    "model_version": version,
    "run_id": best.run_id,
    "scored_at": pd.Timestamp.now(tz="UTC").tz_localize(None),
})
(spark.createDataFrame(predictions).write.mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(T("ml_predictions")))
print(f"Transacciones puntuadas: {len(predictions):,} · alertas: {predictions['alerta'].sum():,} "
      f"({predictions['alerta'].mean():.1%})")
display(spark.table(T("ml_predictions")).orderBy(F.desc("score")).limit(10))

Acá usamos pandas porque son pocos miles de filas. Con millones, el mismo modelo se aplica en paralelo con Spark mediante `mlflow.pyfunc.spark_udf(spark, model_uri)`: el registro y la firma son los mismos, sólo cambia quién ejecuta la predicción.

## 4. Una semana después: llegan los contracargos

El área de riesgo confirma qué compras del 7 de marzo terminaron en contracargo. Ahora podemos medir cómo anduvo el modelo **en producción** y compararlo con lo que prometía en test. Eso es **monitoreo**.

In [ ]:
live = predictions.merge(nuevas[["transaction_id", "chargeback"]], on="transaction_id")
live_metrics = evaluate(live["score"].to_numpy(), live["chargeback"], threshold, "nuevas")
test_metrics = {k.replace("metrics.", ""): v for k, v in client.get_run(best.run_id).data.metrics.items() if k.startswith("test_")}
monitor = pd.DataFrame({
    "test (al entrenar)": {k.replace("test_", ""): v for k, v in test_metrics.items()},
    "nuevas (producción)": {k.replace("nuevas_", ""): v for k, v in live_metrics.items()},
}).loc[["pr_auc", "roc_auc", "precision", "recall", "f1", "accuracy", "alertas"]]
display(monitor.round(3).reset_index(names="métrica"))

In [ ]:
test = load_split(spark, config, "test")
test_scores = champion.predict_proba(test[features])[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
bins = np.linspace(0, 1, 26)
axes[0].hist(test_scores, bins=bins, alpha=0.6, density=True, label="test", color="#2563eb")
axes[0].hist(scores, bins=bins, alpha=0.6, density=True, label="nuevas", color="#f59e0b")
axes[0].axvline(threshold, color="black", ls="--", label="umbral")
axes[0].set_title("Distribución de scores")
axes[0].set_xlabel("Probabilidad de contracargo")
axes[0].legend()
drift = pd.DataFrame({
    "test": [test["amount"].mean(), test["is_night"].mean() * 100, test["chargeback"].mean() * 100, (test_scores >= threshold).mean() * 100],
    "nuevas": [nuevas["amount"].mean(), nuevas["is_night"].mean() * 100, nuevas["chargeback"].mean() * 100, (scores >= threshold).mean() * 100],
}, index=["importe promedio", "% de noche", "% contracargos", "% alertas"])
(drift.div(drift["test"], axis=0) * 100).plot.barh(ax=axes[1], color=["#2563eb", "#f59e0b"])
axes[1].axvline(100, color="black", ls="--")
axes[1].set_title("Nuevas vs. test (test = 100)")
fig.tight_layout()
plt.show()
display(drift.round(2).reset_index(names="indicador"))

## 5. ¿Y si cambian los datos?

En nuestro caso los datos nuevos se parecen a los de entrenamiento. Simulemos lo contrario: una campaña de electrónica hace que las compras sean más caras y que se paguen más por transferencia. No necesitamos etiquetas para notar que algo cambió: alcanza con mirar las entradas y los scores.

In [ ]:
campaign = nuevas.copy()
rng = np.random.default_rng(7)
campaign["amount"] = np.minimum(campaign["amount"] * 1.3, 2001.0)
switch = rng.random(len(campaign)) < 0.4
campaign.loc[switch, "payment_channel"] = "transfer"
campaign["amount_to_price"] = (campaign["amount"] / campaign["price"]).round(4)
campaign_scores = champion.predict_proba(campaign[features])[:, 1]

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.hist(scores, bins=bins, alpha=0.6, density=True, label="nuevas", color="#f59e0b")
ax.hist(campaign_scores, bins=bins, alpha=0.6, density=True, label="con campaña (simulado)", color="#dc2626")
ax.axvline(threshold, color="black", ls="--", label="umbral")
ax.set_xlabel("Probabilidad de contracargo")
ax.set_title("Drift: la distribución de scores se corre")
ax.legend()
plt.show()
print(f"% de alertas: nuevas {(scores >= threshold).mean():.1%} → con campaña {(campaign_scores >= threshold).mean():.1%}")
print(f"Importe promedio: {nuevas['amount'].mean():,.0f} → {campaign['amount'].mean():,.0f} · "
      f"% transferencias: {(nuevas['payment_channel'] == 'transfer').mean():.0%} → {(campaign['payment_channel'] == 'transfer').mean():.0%}")

**Para observar:** cuando los datos cambian (otro mix de medios de pago, una campaña con importes distintos, fraudes nuevos) hay **drift**. Se nota primero en las distribuciones de entrada y de scores, que se pueden vigilar en el momento, y recién después en las métricas, que sólo se conocen cuando llegan las etiquetas. Con la campaña el modelo dispara muchas más alertas: el equipo de riesgo no daría abasto, y no sabemos todavía si esas alertas son correctas. Un cambio así es la señal para **revisar el umbral o reentrenar** con datos recientes, registrar una nueva versión y, si es mejor, moverle el alias `champion`.

## Resumen

- El **Model Registry** da nombre, versiones, alias y linaje a los modelos; se consumen por alias, no por run.
- El **umbral** es una decisión de negocio que se guarda junto al modelo pero fuera de él.
- La **inferencia batch** escribe predicciones con trazabilidad: modelo, versión, run y umbral de cada decisión.
- El **monitoreo** compara lo que el modelo prometía con lo que pasa en producción; las etiquetas llegan tarde, las distribuciones se pueden mirar enseguida.